[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gnoejh/soc3050code/blob/main/projects2026_arm/_notebooks/SOC3050_ARM.ipynb)

# SOC3050 - ARM edition: the Colab workbench

One notebook for the whole STM32 half of the course. It gives you the **build
half** of the toolchain on a machine you do not have to install anything on:
a real `arm-none-eabi-gcc`, the real lesson sources, and the real ELF, sections,
memory map and disassembly that the Part 0 slides quote.

**Slides:** <https://gnoejh.github.io/soc3050code/>

## What this can and cannot do

| | |
|---|---|
| YES | Compile and link real bare-metal STM32 firmware |
| YES | Show sections, the memory map, the vector table, Thumb-2 disassembly |
| YES | Run the code-reading experiments from lessons 01, 02 and 03 |
| NO  | Flash a board - Colab has no USB |
| NO  | Run the firmware - the simulators (Renode, Wokwi) are a local step |

So Colab replaces *installing the toolchain*, not the lab. Everything below is
the same command you would run on your own machine; only the paths differ.

## How to use it

Run the cells in order. **Runtime -> Run all** does the whole thing in about a
minute. The runtime is thrown away when you close the tab, so Step 1 and
Step 2 have to be re-run in a new session.

---
## Step 1 - the toolchain

Colab runs Ubuntu, so the cross-compiler is one `apt-get` away. Three packages
matter and the third is the one people forget:

- `gcc-arm-none-eabi` - the compiler
- `binutils-arm-none-eabi` - `objdump`, `size`, `objcopy`, `nm`
- `libnewlib-arm-none-eabi` - the C library **and** `nano.specs` / `nosys.specs`,
  without which the link fails on `--specs=nano.specs` alone

Ubuntu's package is an older release than the xPack build the course vendors
(15.2.1). Everything Part 0 teaches is the same in both; exact byte counts on
the slides may differ by a few bytes, which is itself worth noticing.

In [ ]:
%%bash
set -e
apt-get -qq update
apt-get -qq install -y gcc-arm-none-eabi binutils-arm-none-eabi libnewlib-arm-none-eabi > /dev/null
echo
arm-none-eabi-gcc --version | head -1
arm-none-eabi-objdump --version | head -1
echo
echo "specs files found:"
find / -name 'nano.specs' 2>/dev/null | head -3

---
## Step 2 - the lesson sources

Two things get cloned:

1. **This repository**, but only `projects2026_arm/`. A full clone is about a
   gigabyte, almost all of it a vendored toolchain and simulator that nothing
   here reads, so this uses `--filter=blob:none` with a sparse checkout - the
   same trick the GitHub Pages workflow uses.
2. **The CMSIS headers.** `stm32c031xx.h` is ST's, `core_cm0plus.h` is ARM's,
   and neither is committed to this repository - they are other people's code
   under their own licences, fetched rather than copied.

In [ ]:
%%bash
set -e
cd /content

# 1. the course tree, projects2026_arm/ only
if [ ! -d soc3050code ]; then
  git clone --filter=blob:none --sparse -b main \
      https://github.com/gnoejh/soc3050code.git
  git -C soc3050code sparse-checkout set projects2026_arm
fi

# 2. the vendor headers (shallow: we want Include/, not the history)
[ -d cmsis-device-c0 ] || git clone --depth 1 -q \
    https://github.com/STMicroelectronics/cmsis-device-c0.git
[ -d CMSIS_6 ]        || git clone --depth 1 -q \
    https://github.com/ARM-software/CMSIS_6.git

echo
ls soc3050code/projects2026_arm
echo
echo "the target we will build:"
ls -l soc3050code/projects2026_arm/_spike/c031c6/*.c \
      soc3050code/projects2026_arm/_spike/c031c6/link.ld

---
## Step 3 - build a real firmware  *(lesson 02)*

This is the whole build. Three files - `startup.c`, `main.c`, `link.ld` - and
no IDE, no build system, no vendor code generator.

Read the flags. Every one of them is on a slide:

| flag | why |
|---|---|
| `-mcpu=cortex-m0plus -mthumb` | which core, and Thumb encoding only |
| `-mfloat-abi=soft` | this chip has no FPU |
| `-DSTM32C031xx` | tells ST's header which chip's register map to define |
| `-ffunction-sections -fdata-sections` + `-Wl,--gc-sections` | one section per function, then drop the unreferenced ones |
| `--specs=nano.specs` | newlib-nano: a `printf` that fits |
| `--specs=nosys.specs` | stub out the syscalls a bare-metal chip has no OS for |
| `-T link.ld` | **the memory map** - without it the linker has no idea where flash is |
| `-Wl,--print-memory-usage` | the FLASH/RAM table printed below |

In [ ]:
%%bash
set -e
cd /content/soc3050code/projects2026_arm/_spike/c031c6

arm-none-eabi-gcc \
  -mcpu=cortex-m0plus -mthumb -mfloat-abi=soft \
  -DSTM32C031xx -Os -g3 -std=c11 -Wall -Wextra \
  -ffunction-sections -fdata-sections \
  -I/content/cmsis-device-c0/Include \
  -I/content/CMSIS_6/CMSIS/Core/Include \
  startup.c main.c \
  --specs=nano.specs --specs=nosys.specs \
  -T link.ld -Wl,--gc-sections -Wl,-Map=Main.map \
  -Wl,--print-memory-usage -o Main.elf

arm-none-eabi-objcopy -O ihex   Main.elf Main.hex
arm-none-eabi-objcopy -O binary Main.elf Main.bin
echo
ls -l Main.elf Main.hex Main.bin

### Where it went

`--print-memory-usage` above is the linker checking your program against the
`MEMORY` block in `link.ld`:

```
FLASH (rx) : ORIGIN = 0x08000000, LENGTH = 32K
RAM   (rw) : ORIGIN = 0x20000000, LENGTH = 12K
```

Those four numbers come from the datasheet. Get `LENGTH` wrong and the build
still succeeds - the failure arrives later, as a stack that walks off the end
of memory.

---
## Step 4 - your program is not one blob  *(lesson 02, slides 4-5)*

`size -A` gives the per-section split. Read the **addresses**, not just the
sizes: `0x08...` is flash, `0x20...` is SRAM.

Watch `.data`. It is listed once, but it exists twice - the values in flash so
they survive power-off, the variables in RAM so they can be written. The
`objdump -h` output below shows both: `LMA` is where the bytes are stored,
`VMA` is where they will live. For `.data` those two differ, and copying
between them is the first thing `Reset_Handler` does.

In [ ]:
%%bash
cd /content/soc3050code/projects2026_arm/_spike/c031c6
echo "=== arm-none-eabi-size -A ==============================================="
arm-none-eabi-size -A Main.elf
echo
echo "=== section headers: note VMA vs LMA on .data =========================="
arm-none-eabi-objdump -h Main.elf | grep -E 'Idx|isr_vector|\.text|\.rodata|\.data|\.bss'

In [ ]:
%%bash
cd /content/soc3050code/projects2026_arm/_spike/c031c6
echo "the linker-invented symbols Reset_Handler reads:"
arm-none-eabi-nm Main.elf | grep -E ' (_sidata|_sdata|_edata|_sbss|_ebss|_estack)$'

---
## Step 5 - the first eight bytes  *(lesson 00)*

A Cortex-M does not start at a reset *instruction*. It starts by **reading two
words from address 0**: the initial stack pointer, then the address of
`Reset_Handler`. That is the vector table, and it is the first thing in flash.

The cell below reads them straight out of the raw binary and checks them
against the symbol table. The reset address will end in an **odd** number -
bit 0 set is the Thumb bit, not part of the address.

In [ ]:
import struct, subprocess, pathlib
d = pathlib.Path('/content/soc3050code/projects2026_arm/_spike/c031c6')
raw = (d / 'Main.bin').read_bytes()
sp, reset = struct.unpack_from('<II', raw, 0)

print('Main.bin, first 8 bytes :', ' '.join('%02X' % b for b in raw[:8]))
print('initial stack pointer   : 0x%08X' % sp)
print('reset vector            : 0x%08X   (bit 0 = %d, the Thumb bit)'
      % (reset, reset & 1))
print('so execution starts at  : 0x%08X' % (reset & ~1))
print()

syms = subprocess.run(['arm-none-eabi-nm', str(d / 'Main.elf')],
                      capture_output=True, text=True).stdout
for line in syms.splitlines():
    if line.endswith(' Reset_Handler') or line.endswith(' _estack'):
        print('symbol table            :', line)

In [ ]:
%%bash
cd /content/soc3050code/projects2026_arm/_spike/c031c6
echo "=== the vector table, as stored bytes =================================="
arm-none-eabi-objdump -s -j .isr_vector Main.elf | head -12
echo
echo "=== Reset_Handler ======================================================"
arm-none-eabi-objdump -d --disassemble=Reset_Handler Main.elf | tail -n +6

---
## Step 6 - Thumb-2, one function at a time  *(lesson 01)*

The fastest way to learn an instruction set is to write C and read what the
compiler did with it. `-S` stops after compiling, so you get assembly instead
of an object file.

**Edit the C below and re-run.** Things worth trying:

- change `-Os` to `-O0` and watch the same function triple in length
- make `n` a `uint8_t` and look for the `UXTB` that appears
- replace `* 10` with `* 8`, then `* 7` - the compiler will not use a multiply
  for either of them
- return `a / b` and see a whole function get **called**: Cortex-M0+ has no
  divide instruction

In [ ]:
%%writefile /content/demo.c
#include <stdint.h>

uint32_t scale(uint32_t n)
{
    return n * 10u;
}

uint32_t sum_to(uint32_t n)
{
    uint32_t total = 0;
    for (uint32_t i = 1; i <= n; i++)
        total += i;
    return total;
}

In [ ]:
%%bash
cd /content
arm-none-eabi-gcc -mcpu=cortex-m0plus -mthumb -Os -std=c11 -S demo.c -o demo.s
grep -v '^\s*\.' demo.s | grep -v '^\s*$'

---
## Step 7 - `volatile`, and why `count++` is not one thing  *(lesson 03)*

Two experiments, both of which produce a bug that is invisible in the C.

**7a. Without `volatile`** the compiler is entitled to assume nothing else
changes the variable, so it reads it once and loops forever on a stale value.
This is the classic "waiting for the ISR flag" hang - and it only appears with
optimisation on, which is why it survives testing at `-O0`.

In [ ]:
%%writefile /content/vol.c
#include <stdint.h>

static          uint32_t plain_flag;
static volatile uint32_t volatile_flag;

void wait_plain(void)    { while (!plain_flag)    { } }
void wait_volatile(void) { while (!volatile_flag) { } }

In [ ]:
%%bash
cd /content
arm-none-eabi-gcc -mcpu=cortex-m0plus -mthumb -O2 -std=c11 -S vol.c -o vol.s
echo "=== wait_plain ========================================================="
sed -n '/^wait_plain:/,/\.size\twait_plain/p'       vol.s | grep -v '^\s*\.'
echo "=== wait_volatile ======================================================"
sed -n '/^wait_volatile:/,/\.size\twait_volatile/p' vol.s | grep -v '^\s*\.'

`wait_plain` loads once and then branches to itself forever. `wait_volatile`
has an `LDR` **inside** the loop - it re-reads memory every pass, which is the
whole point of the keyword.

**7b. A read-modify-write is three instructions.** `volatile` fixes visibility;
it does nothing about atomicity. If an interrupt fires between the load and the
store, its update is lost - and the window is two instructions wide, so the bug
appears once an hour and never while you are watching.

In [ ]:
%%writefile /content/rmw.c
#include <stdint.h>
volatile uint32_t count;
void bump(void) { count++; }

In [ ]:
%%bash
cd /content
arm-none-eabi-gcc -mcpu=cortex-m0plus -mthumb -O2 -std=c11 -S rmw.c -o rmw.s
sed -n '/^bump:/,/\.size\tbump/p' rmw.s | grep -v '^\s*\.'

Count them: a load, an add, a store. Three chances to be interrupted, for
one line of C. The fix is a critical section - disable interrupts, or on a
core that has them, `LDREX`/`STREX`. Cortex-M0+ does **not** have those, which
is why the STM32C0 lessons use `__disable_irq()` / `__enable_irq()`.

---
## Where to go next

| | |
|---|---|
| The slides | <https://gnoejh.github.io/soc3050code/> |
| Build locally | `git clone https://github.com/gnoejh/soc3050code` - the repository vendors its own toolchain |
| Run the firmware | Renode or Wokwi, both a local step; see `projects2026_arm/_spike/README.md` |
| The chip | [STM32C0x1 reference manual](https://www.st.com/resource/en/reference_manual/rm0490-stm32c0x1-advanced-armbased-32bit-mcus-stmicroelectronics.pdf) |

To keep anything you changed here, **File -> Save a copy in Drive** before you
close the tab.